# GPU
:label:`sec_use_gpu`

## 2024 年更新：多平台加速器支援

PyTorch 2.0+ 現在支援多種加速器：
- **CUDA** - NVIDIA GPU (最成熟，功能最全)
- **MPS** - Apple Silicon (M1/M2/M3/M4 晶片)
- **ROCm** - AMD GPU
- **XLA** - Google TPU

本章節已更新以支援所有主流平台。

### 設備選擇最佳實踐

建議使用 `.to(device)` 方法而非 `.cuda()` 以獲得更好的可移植性：

```python
# 推薦寫法
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
data = data.to(device)

# 避免硬編碼
model.cuda()  # 只能在 CUDA 設備上工作
```

---

在 :numref:`tab_intro_decade`中，
我們回顧了過去20年計算能力的快速增長。
簡而言之，自2000年以來，GPU性能每十年增長1000倍。

本節，我們將討論如何利用這種計算性能進行研究。
首先是如何使用單個GPU，然後是如何使用多個GPU和多個伺服器（具有多個GPU）。

我們先看看如何使用單個NVIDIA GPU進行計算。
首先，確保至少安裝了一個NVIDIA GPU。
然後，下載[NVIDIA驅動和CUDA](https://developer.nvidia.com/cuda-downloads)
並按照提示設置適當的路徑。
當這些準備工作完成，就可以使用`nvidia-smi`命令來(**查看顯卡信息。**)

In [1]:
!nvidia-smi

Fri Aug 18 06:58:06 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 470.161.03   Driver Version: 470.161.03   CUDA Version: 11.7     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla V100-SXM2...  Off  | 00000000:00:1B.0 Off |                    0 |
| N/A   41C    P0    42W / 300W |      0MiB / 16160MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+


|   1  Tesla V100-SXM2...  Off  | 00000000:00:1C.0 Off |                    0 |
| N/A   44C    P0   113W / 300W |   1456MiB / 16160MiB |     53%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
|   2  Tesla V100-SXM2...  Off  | 00000000:00:1D.0 Off |                    0 |
| N/A   43C    P0   120W / 300W |   1358MiB / 16160MiB |     55%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
|   3  Tesla V100-SXM2...  Off  | 00000000:00:1E.0 Off |                    0 |
| N/A   42C    P0    47W / 300W |      0MiB / 16160MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                        

在PyTorch中，每個數組都有一個設備（device），
我們通常將其稱為環境（context）。
默認情況下，所有變量和相關的計算都分配給CPU。
有時環境可能是GPU。
當我們跨多個伺服器部署作業時，事情會變得更加棘手。
通過智能地將數組分配給環境，
我們可以最大限度地減少在設備之間傳輸數據的時間。
例如，當在帶有GPU的伺服器上訓練神經網路時，
我們通常希望模型的參數在GPU上。


要運行此部分中的程式，至少需要兩個GPU。
注意，對大多數桌面計算機來說，這可能是奢侈的，但在雲中很容易獲得。
例如可以使用AWS EC2的多GPU實例。
本書的其他章節大都不需要多個GPU，
而本節只是為了展示數據如何在不同的設備之間傳遞。

## [**計算設備**]

我們可以指定用於存儲和計算的設備，如CPU和GPU。
默認情況下，張量是在記憶體中創建的，然後使用CPU計算它。


在PyTorch中，CPU和GPU可以用`torch.device('cpu')`
和`torch.device('cuda')`表示。
應該注意的是，`cpu`設備意味著所有物理CPU和記憶體，
這意味著PyTorch的計算將嘗試使用所有CPU核心。
然而，`gpu`設備只代表一個卡和相應的顯存。
如果有多個GPU，我們使用`torch.device(f'cuda:{i}')`
來表示第$i$塊GPU（$i$從0開始）。
另外，`cuda:0`和`cuda`是等價的。


In [2]:
import torch
from torch import nn

torch.device('cpu'), torch.device('cuda'), torch.device('cuda:1')

(device(type='cpu'), device(type='cuda'), device(type='cuda', index=1))

我們可以(**查詢可用gpu的數量。**)


In [3]:
torch.cuda.device_count()

2

現在我們定義了兩個方便的函數，
[**這兩個函數允許我們在不存在所需所有GPU的情況下運行程式。**]


In [ ]:
def try_gpu(i=0):  #@save
    """如果存在，則返回 gpu(i)，否則返回 cpu()
    
    支援的加速器:
    - CUDA (NVIDIA GPU)
    - MPS (Apple Silicon)
    """
    # 支援 Apple Silicon MPS (PyTorch 1.12+)
    if i == 0 and torch.backends.mps.is_available():
        return torch.device('mps')
    # 支援 NVIDIA CUDA
    if torch.cuda.device_count() >= i + 1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')

def try_all_gpus():  #@save
    """返回所有可用的GPU，如果沒有GPU，則返回[cpu(),]"""
    devices = [torch.device(f'cuda:{i}')
             for i in range(torch.cuda.device_count())]
    # 添加 MPS 支援
    if not devices and torch.backends.mps.is_available():
        devices = [torch.device('mps')]
    return devices if devices else [torch.device('cpu')]

def get_available_devices():  #@save
    """檢測並列出所有可用的計算設備"""
    devices = []
    
    # CPU 總是可用
    devices.append('cpu')
    
    # 檢測 CUDA
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            devices.append(f'cuda:{i}')
    
    # 檢測 MPS
    if torch.backends.mps.is_available():
        devices.append('mps')
    
    return devices

# 測試函數
print("可用設備:", get_available_devices())
print("try_gpu():", try_gpu())
print("try_gpu(10):", try_gpu(10))
print("try_all_gpus():", try_all_gpus())

## 張量與GPU

我們可以[**查詢張量所在的設備。**]
默認情況下，張量是在CPU上創建的。


In [5]:
x = torch.tensor([1, 2, 3])
x.device

device(type='cpu')

需要注意的是，無論何時我們要對多個項目進行操作，
它們都必須在同一個設備上。
例如，如果我們對兩個張量求和，
我們需要確保兩個張量都位於同一個設備上，
否則框架將不知道該在哪裡存儲結果，甚至不知道該在哪裡執行計算。

### [**存儲在GPU上**]

有幾種方法可以在GPU上存儲張量。
例如，我們可以在創建張量時指定存儲設備。接
下來，我們在第一個`gpu`上創建張量變量`X`。
在GPU上創建的張量只消耗這個GPU的顯存。
我們可以使用`nvidia-smi`命令查看顯存使用情況。
一般來說，我們需要確保不創建超過GPU顯存限制的數據。


In [6]:
X = torch.ones(2, 3, device=try_gpu())
X

tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:0')

假設我們至少有兩個GPU，下面的程式碼將在(**第二個GPU上創建一個隨機張量。**)


In [7]:
Y = torch.rand(2, 3, device=try_gpu(1))
Y

tensor([[0.4860, 0.1285, 0.0440],
        [0.9743, 0.4159, 0.9979]], device='cuda:1')

### 複製

如果我們[**要計算`X + Y`，我們需要決定在哪裡執行這個操作**]。
例如，如 :numref:`fig_copyto`所示，
我們可以將`X`傳輸到第二個GPU並在那裡執行操作。
*不要*簡單地`X`加上`Y`，因為這會導致異常，
運行時引擎不知道該怎麼做：它在同一設備上找不到數據會導致失敗。
由於`Y`位於第二個GPU上，所以我們需要將`X`移到那裡，
然後才能執行相加運算。

![複製數據以在同一設備上執行操作](../img/copyto.svg)
:label:`fig_copyto`


In [ ]:
# 使用 .to(device) 方法更具可移植性
Z = X.to(try_gpu(1))
print(X)
print(Z)

[**現在數據在同一個GPU上（`Z`和`Y`都在），我們可以將它們相加。**]


In [9]:
Y + Z

tensor([[1.4860, 1.1285, 1.0440],
        [1.9743, 1.4159, 1.9979]], device='cuda:1')

假設變量`Z`已經存在於第二個GPU上。
如果我們還是調用`Z.cuda(1)`會發生什麼？
它將返回`Z`，而不会複製並分配新記憶體。


In [ ]:
# 檢查是否會創建新的張量副本
Z.to(try_gpu(1)) is Z

### 旁注

人們使用GPU進行機器學習，因為單個GPU相對運行速度快。
但是在設備（CPU、GPU和其他機器）之間傳輸數據比計算慢得多。
這也使得並行化變得更加困難，因為我們必須等待數據被發送（或者接收），
然後才能繼續進行更多的操作。
這就是為什麼拷貝操作要格外小心。
根據經驗，多個小操作比一個大操作糟糕得多。
此外，一次執行幾個操作比程式碼中散布的許多單個操作要好得多。
如果一個設備必須等待另一個設備才能執行其他操作，
那麼這樣的操作可能會阻塞。
這有點像排隊訂購咖啡，而不像通過電話預先訂購：
當客人到店的时候，咖啡已經準備好了。

最后，當我們打印張量或將張量轉換為NumPy格式時，
如果數據不在記憶體中，框架會首先將其複製到記憶體中，
這會導致額外的傳輸開銷。
更糟糕的是，它現在受制於全局解釋器鎖，使得一切都得等待Python完成。

## [**神經網路與GPU**]

類似地，神經網路模型可以指定設備。
下面的程式碼將模型參數放在GPU上。


In [11]:
net = nn.Sequential(nn.Linear(3, 1))
net = net.to(device=try_gpu())

### Apple Silicon (MPS) 注意事項

✅ **支援的操作**：
- 基本張量運算（加減乘除、矩陣乘法等）
- 常見神經網絡層（Linear, Conv2d, BatchNorm 等）
- 自動微分和反向傳播
- 大多數深度學習操作

⚠️ **限制**：
- 部分高級操作可能不支援（如某些稀疏張量操作）
- 性能可能不如 CUDA（但通常比 CPU 快 5-10 倍）
- 某些操作會自動回退到 CPU
- 不支援多 GPU 操作（MPS 只能使用單個設備）

🔧 **最佳實踐**：
```python
# 檢查 MPS 可用性
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print("MPS 設備可用")
else:
    device = torch.device('cpu')
    print("MPS 設備不可用，使用 CPU")

# 使用 MPS 訓練模型
model = model.to(device)
data = data.to(device)
```

**性能建議**：
- 開發和調試時在 MPS 上測試（Apple Silicon Mac）
- 大規模訓練時使用 CUDA（雲端 GPU 服務器）
- 始終確保數據和模型在同一設備上
- 注意內存管理，MPS 與系統共享記憶體

In [ ]:
# MPS 設備檢測和使用示例
print("=== 設備可用性檢測 ===")
print(f"CUDA 可用: {torch.cuda.is_available()}")
print(f"MPS 可用: {torch.backends.mps.is_available()}")
print(f"MPS 已構建: {torch.backends.mps.is_built()}")

if torch.backends.mps.is_available():
    print("\n=== MPS 設備測試 ===")
    mps_device = torch.device('mps')
    
    # 在 MPS 上創建張量
    x_mps = torch.randn(3, 3, device=mps_device)
    y_mps = torch.randn(3, 3, device=mps_device)
    
    # 矩陣運算
    z_mps = torch.mm(x_mps, y_mps)
    
    print(f"張量設備: {x_mps.device}")
    print(f"計算結果形狀: {z_mps.shape}")
    print(f"計算結果:\n{z_mps}")
else:
    print("\n注意: MPS 設備在此系統上不可用")
    print("這是正常的，如果你不是在 Apple Silicon Mac 上運行的話")

在接下來的幾章中，
我們將看到更多關於如何在GPU上運行模型的例子，
因為它們將變得更加計算密集。

當輸入為GPU上的張量時，模型將在同一GPU上計算結果。


In [12]:
net(X)

tensor([[-0.4275],
        [-0.4275]], device='cuda:0', grad_fn=<AddmmBackward0>)

讓我們(**確認模型參數存儲在同一個GPU上。**)


In [13]:
net[0].weight.data.device

device(type='cuda', index=0)

## 跨平台設備管理示例

以下是一個完整的跨平台示例，展示如何編寫在 CUDA、MPS 和 CPU 上都能運行的代碼：

In [ ]:
# 跨平台設備選擇函數
def get_device():
    """自動選擇最佳可用設備"""
    if torch.cuda.is_available():
        return torch.device('cuda')
    elif torch.backends.mps.is_available():
        return torch.device('mps')
    else:
        return torch.device('cpu')

# 獲取設備
device = get_device()
print(f"使用設備: {device}")

# 創建模型和數據（跨平台兼容）
model = nn.Sequential(
    nn.Linear(10, 5),
    nn.ReLU(),
    nn.Linear(5, 2)
).to(device)

# 創建輸入數據
input_data = torch.randn(4, 10).to(device)

# 前向傳播
output = model(input_data)
print(f"輸出形狀: {output.shape}")
print(f"輸出設備: {output.device}")

# 驗證設備一致性
print(f"\n設備一致性檢查:")
print(f"模型在: {next(model.parameters()).device}")
print(f"數據在: {input_data.device}")
print(f"輸出在: {output.device}")

## 小結

* 我們可以指定用於存儲和計算的設備，例如 CPU、NVIDIA GPU (CUDA) 或 Apple Silicon (MPS)。默認情況下，數據在主記憶體中創建，然後使用 CPU 進行計算。
* 深度學習框架要求計算的所有輸入數據都在同一設備上，無論是 CPU、CUDA 還是 MPS。
* 不小心移動數據可能會顯著降低性能。一個典型的錯誤如下：計算 GPU 上每個小批量的損失，並在命令行中將其報告給用戶（或將其記錄在 NumPy `ndarray` 中）時，將觸發全局解釋器鎖，從而使所有 GPU 阻塞。最好為 GPU 內部的日誌分配記憶體，並且只移動較大的日誌。
* 使用 `.to(device)` 方法比直接使用 `.cuda()` 更具可移植性，能在不同平台間無縫切換。
* PyTorch 2.0+ 支援多種加速器後端，包括 CUDA、MPS、ROCm 等，編寫跨平台代碼時應考慮設備兼容性。

## 練習

1. 嘗試一個計算量更大的任務，比如大矩陣的乘法，看看CPU和GPU之間的速度差異。再試一個計算量很小的任務呢？
1. 我們應該如何在GPU上讀寫模型參數？
1. 測量計算1000個$100 \times 100$矩陣的矩陣乘法所需時間，並記錄輸出矩陣的Frobenius範數，一次記錄一個結果，而不是在GPU上保存日誌並僅傳輸最終結果。
1. 測量同時在兩個GPU上執行兩個矩陣乘法與在一個GPU上按順序執行兩個矩陣乘法所需時間。提示：應該看到近乎線性的縮放。
1. **(新增)** 如果你有 Apple Silicon Mac，比較在 MPS、CPU 上訓練一個小型神經網絡的速度差異。
1. **(新增)** 嘗試將模型在 CUDA、MPS 和 CPU 之間移動，觀察數據傳輸的開銷。

練習一：

1. 嘗試一個計算量更大的任務，比如大矩陣的乘法，看看CPU和GPU之間的速度差異。再試一個計算量很小的任務呢？

我的回答：





以下是比較CPU和GPU在不同計算量任務上的性能差異：

```python
import torch
import time

def benchmark(func, *args, **kwargs):
    """測量函數運行時間"""
    start = time.time()
    result = func(*args, **kwargs)
    end = time.time()
    return end - start, result

def test_device(size, device):
    """在指定設備上進行矩陣運算"""
    # 創建矩陣
    a = torch.randn(size, size, device=device)
    b = torch.randn(size, size, device=device)
    
    # 確保GPU完成前面的操作
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    # 計時矩陣乘法
    time_taken, _ = benchmark(torch.mm, a, b)
    
    return time_taken

# 1. 大規模計算（大矩陣乘法）
print("大規模矩陣乘法測試 (2000x2000):")
for device in [torch.device('cpu'), torch.device('cuda')]:
    time_taken = test_device(2000, device)
    print(f"{device} 用時: {time_taken:.4f} 秒")

# 2. 小規模計算（小矩陣乘法）
print("\n小規模矩陣乘法測試 (10x10):")
for device in [torch.device('cpu'), torch.device('cuda')]:
    time_taken = test_device(10, device)
    print(f"{device} 用時: {time_taken:.4f} 秒")

# 3. 重複小規模計算（測試數據傳輸開銷）
print("\n重複小規模計算測試 (10x10, 1000次):")
for device in [torch.device('cpu'), torch.device('cuda')]:
    start = time.time()
    for _ in range(1000):
        _ = test_device(10, device)
    total_time = time.time() - start
    print(f"{device} 用時: {total_time:.4f} 秒")
```



觀察結果：

1. 大規模計算：
- GPU 明顯快於 CPU
- 優勢可能達到 10-100 倍
- GPU 的並行計算能力得到充分發揮

2. 小規模計算：
- CPU 可能快於 GPU
- GPU 的數據傳輸開銷成為主要瓶頸
- 並行計算優勢無法體現

3. 結論：
- GPU 適合大規模並行計算
- 小規模計算應該留在 CPU 上
- 需要考慮數據傳輸開銷

改進版本（添加更多測試）：
```python
def comprehensive_benchmark():
    sizes = [10, 100, 500, 1000, 2000]
    iterations = [1, 10, 100]
    
    results = {}
    
    for size in sizes:
        for iters in iterations:
            print(f"\n測試矩陣大小: {size}x{size}, 重複次數: {iters}")
            
            for device in [torch.device('cpu'), torch.device('cuda')]:
                start = time.time()
                for _ in range(iters):
                    _ = test_device(size, device)
                total_time = time.time() - start
                
                key = f"size_{size}_iters_{iters}_{device.type}"
                results[key] = total_time
                
                print(f"{device} 總用時: {total_time:.4f} 秒")
                print(f"{device} 平均用時: {total_time/iters:.4f} 秒")
    
    return results

# 運行綜合測試
results = comprehensive_benchmark()

# 分析最佳使用場景
for size in [10, 100, 500, 1000, 2000]:
    cpu_time = results[f"size_{size}_iters_1_cpu"]
    gpu_time = results[f"size_{size}_iters_1_cuda"]
    speedup = cpu_time / gpu_time
    print(f"\n矩陣大小 {size}x{size}:")
    print(f"GPU 加速比: {speedup:.2f}x")
```



使用建議：

1. 大規模計算：
- 使用 GPU
- 盡量批量處理
- 減少 CPU-GPU 數據傳輸

2. 小規模計算：
- 留在 CPU 上
- 避免不必要的 GPU 傳輸
- 考慮向量化操作

3. 混合計算：
- 根據計算量動態選擇設備
- 合理安排數據傳輸
- 使用異步操作優化性能



練習二：

2. 我們應該如何在GPU上讀寫模型參數？

我的回答：





以下是在GPU上讀寫模型參數的示例：

````python
import torch
import torch.nn as nn

# 1. 創建模型並移動到GPU
model = nn.Linear(10, 5)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# 2. 讀取參數
print("讀取參數：")
for name, param in model.named_parameters():
    print(f"{name} 在設備: {param.device}")
    print(f"參數值:\n{param.data[:2]}")
    print(f"梯度:\n{param.grad}\n")

# 3. 修改參數
with torch.no_grad():  # 禁用梯度計算
    model.weight.data = torch.randn_like(model.weight.data)
    model.bias.data.zero_()  # 將偏置設為0

# 4. 保存和加載
# 保存到文件
torch.save(model.state_dict(), 'model_gpu.pth')

# 加載到GPU
new_model = nn.Linear(10, 5).to(device)
new_model.load_state_dict(torch.load('model_gpu.pth'))

# 5. 在不同設備間移動
# GPU -> CPU
cpu_model = model.cpu()
# CPU -> GPU
gpu_model = cpu_model.cuda()

# 6. 批量操作參數
for param in model.parameters():
    with torch.no_grad():
        param.data *= 2  # 將所有參數翻倍
````



最佳實踐：

1. 參數初始化：
````python
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_normal_(m.weight)
        nn.init.zeros_(m.bias)

model = nn.Sequential(nn.Linear(10, 5), nn.ReLU())
model.apply(init_weights)  # 應用初始化
model = model.to(device)  # 移動到GPU
````

2. 參數更新：
````python
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
for epoch in range(num_epochs):
    optimizer.zero_grad()
    output = model(input_data.to(device))
    loss = criterion(output, target.to(device))
    loss.backward()
    optimizer.step()
````

3. 檢查點保存和加載：
````python
# 保存檢查點
checkpoint = {
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'epoch': epoch
}
torch.save(checkpoint, 'checkpoint.pth')

# 加載檢查點
checkpoint = torch.load('checkpoint.pth')
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
````

注意事項：
1. 確保數據和模型在同一設備上
2. 使用 `with torch.no_grad()` 修改參數
3. 保存時注意設備兼容性
4. 批量操作時注意內存使用
5. 使用適當的初始化方法

這樣可以高效且安全地在GPU上管理模型參數。


練習三：

3. 測量計算1000個$100 \times 100$矩陣的矩陣乘法所需時間，並記錄輸出矩陣的Frobenius範數，一次記錄一個結果，而不是在GPU上保存日誌並僅傳輸最終結果。

我的回答：







以下是測量矩陣乘法時間和Frobenius範數的代碼：

`````python
import torch
import time

def benchmark_matrix_mult():
    # 設置設備
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # 初始化結果列表（在CPU上）
    norms = []
    times = []
    
    # 執行1000次計算
    for i in range(1000):
        # 在GPU上創建矩陣
        a = torch.randn(100, 100, device=device)
        b = torch.randn(100, 100, device=device)
        
        # 計時開始
        start = time.time()
        
        # 矩陣乘法
        c = torch.mm(a, b)
        
        # 確保GPU完成計算
        if device.type == 'cuda':
            torch.cuda.synchronize()
            
        # 計算時間
        time_taken = time.time() - start
        
        # 計算Frobenius範數並移到CPU
        norm = torch.norm(c).cpu().item()
        
        # 記錄結果
        norms.append(norm)
        times.append(time_taken)
        
        # 每100次打印進度
        if (i + 1) % 100 == 0:
            print(f"完成 {i + 1} 次計算")
    
    return times, norms

# 執行測試
print("開始測試...")
times, norms = benchmark_matrix_mult()

# 分析結果
print(f"\n統計結果:")
print(f"平均計算時間: {sum(times)/len(times):.6f} 秒")
print(f"最短時間: {min(times):.6f} 秒")
print(f"最長時間: {max(times):.6f} 秒")
print(f"範數平均值: {sum(norms)/len(norms):.2f}")
print(f"範數標準差: {torch.tensor(norms).std():.2f}")
`````



這個實現的特點：

1. 效率考慮：
- 每次計算後立即記錄結果
- 避免在GPU上累積數據
- 使用 `item()` 高效傳輸單個數值

2. 記憶體管理：
- 每次迭代後釋放GPU記憶體
- 結果保存在CPU上
- 避免大量數據傳輸

3. 同步處理：
- 使用 `torch.cuda.synchronize()` 確保準確計時
- 正確處理GPU異步執行

改進版本（添加更多分析）：
`````python
def detailed_benchmark():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    results = {
        'times': [],
        'norms': [],
        'memory_usage': []
    }
    
    # 記錄初始GPU記憶體
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    
    for i in range(1000):
        if device.type == 'cuda':
            torch.cuda.empty_cache()
        
        # 矩陣計算
        a = torch.randn(100, 100, device=device)
        b = torch.randn(100, 100, device=device)
        
        start = time.time()
        c = torch.mm(a, b)
        
        if device.type == 'cuda':
            torch.cuda.synchronize()
        
        time_taken = time.time() - start
        norm = torch.norm(c).cpu().item()
        
        # 記錄結果
        results['times'].append(time_taken)
        results['norms'].append(norm)
        
        if device.type == 'cuda':
            memory = torch.cuda.max_memory_allocated() / 1024**2  # MB
            results['memory_usage'].append(memory)
        
        if (i + 1) % 100 == 0:
            print(f"完成 {i + 1} 次計算")
            print(f"當前平均時間: {sum(results['times'][-100:])/100:.6f} 秒")
    
    return results

# 執行並分析
results = detailed_benchmark()

# 詳細分析
print("\n詳細統計:")
times = torch.tensor(results['times'])
norms = torch.tensor(results['norms'])

print(f"時間統計:")
print(f"平均: {times.mean():.6f} ± {times.std():.6f} 秒")
print(f"中位數: {times.median():.6f} 秒")
print(f"範數統計:")
print(f"平均: {norms.mean():.2f} ± {norms.std():.2f}")

if 'memory_usage' in results:
    memory = torch.tensor(results['memory_usage'])
    print(f"\nGPU記憶體使用:")
    print(f"平均: {memory.mean():.2f} MB")
    print(f"最大: {memory.max():.2f} MB")
`````



這個改進版本：
1. 監控GPU記憶體使用
2. 提供更詳細的統計分析
3. 實時顯示性能指標
4. 更好的記憶體管理

這樣可以更全面地了解計算性能和資源使用情況。


練習四：

4. 測量同時在兩個GPU上執行兩個矩陣乘法與在一個GPU上按順序執行兩個矩陣乘法所需時間。提示：應該看到近乎線性的縮放。

我的回答：







以下是比較單GPU順序執行和雙GPU並行執行的代碼：

``````python
import torch
import time

def matrix_multiply(size, device):
    """在指定設備上執行矩陣乘法"""
    a = torch.randn(size, size, device=device)
    b = torch.randn(size, size, device=device)
    
    if device.type == 'cuda':
        torch.cuda.synchronize(device)
    
    start = time.time()
    c = torch.mm(a, b)
    
    if device.type == 'cuda':
        torch.cuda.synchronize(device)
        
    time_taken = time.time() - start
    return time_taken, torch.norm(c).item()

def sequential_multiply(size):
    """在單個GPU上順序執行兩次矩陣乘法"""
    device = torch.device('cuda:0')
    
    start = time.time()
    time1, norm1 = matrix_multiply(size, device)
    time2, norm2 = matrix_multiply(size, device)
    total_time = time.time() - start
    
    return total_time, [norm1, norm2]

def parallel_multiply(size):
    """在兩個GPU上並行執行矩陣乘法"""
    device1 = torch.device('cuda:0')
    device2 = torch.device('cuda:1')
    
    # 創建兩個矩陣乘法任務
    start = time.time()
    
    # 在兩個GPU上同時啟動計算
    a1 = torch.randn(size, size, device=device1)
    b1 = torch.randn(size, size, device=device1)
    a2 = torch.randn(size, size, device=device2)
    b2 = torch.randn(size, size, device=device2)
    
    # 並行執行
    c1 = torch.mm(a1, b1)
    c2 = torch.mm(a2, b2)
    
    # 同步兩個GPU
    torch.cuda.synchronize(device1)
    torch.cuda.synchronize(device2)
    
    total_time = time.time() - start
    norm1 = torch.norm(c1).item()
    norm2 = torch.norm(c2).item()
    
    return total_time, [norm1, norm2]

# 測試不同大小的矩陣
sizes = [1000, 2000, 3000]
num_trials = 5

print("比較單GPU順序執行和雙GPU並行執行：")
for size in sizes:
    print(f"\n矩陣大小: {size}x{size}")
    
    # 多次測試取平均
    seq_times = []
    par_times = []
    
    for trial in range(num_trials):
        # 順序執行
        seq_time, seq_norms = sequential_multiply(size)
        seq_times.append(seq_time)
        
        # 並行執行
        par_time, par_norms = parallel_multiply(size)
        par_times.append(par_time)
        
        # 清理GPU緩存
        torch.cuda.empty_cache()
    
    # 計算平均時間
    avg_seq_time = sum(seq_times) / len(seq_times)
    avg_par_time = sum(par_times) / len(par_times)
    speedup = avg_seq_time / avg_par_time
    
    print(f"順序執行平均時間: {avg_seq_time:.4f} 秒")
    print(f"並行執行平均時間: {avg_par_time:.4f} 秒")
    print(f"加速比: {speedup:.2f}x")
``````




這個實現的特點：

1. 測試設計：
- 比較單GPU順序執行和雙GPU並行執行
- 測試多個矩陣大小
- 多次重複測試取平均

2. 性能優化：
- 使用 `torch.cuda.synchronize()` 確保準確計時
- 清理GPU緩存避免影響
- 考慮數據傳輸開銷

3. 結果分析：
- 計算加速比
- 觀察規模效應
- 驗證線性縮放

改進版本（添加更多分析）：
``````python
def benchmark_matrix_operations(sizes=[1000, 2000, 3000], num_trials=5):
    results = {
        'sequential': {},
        'parallel': {}
    }
    
    for size in sizes:
        results['sequential'][size] = {
            'times': [],
            'memory': [],
            'norms': []
        }
        results['parallel'][size] = {
            'times': [],
            'memory': [],
            'norms': []
        }
        
        for trial in range(num_trials):
            # 記錄初始記憶體狀態
            torch.cuda.reset_peak_memory_stats()
            
            # 順序執行
            seq_time, seq_norms = sequential_multiply(size)
            seq_memory = torch.cuda.max_memory_allocated() / 1024**2
            
            results['sequential'][size]['times'].append(seq_time)
            results['sequential'][size]['memory'].append(seq_memory)
            results['sequential'][size]['norms'].append(seq_norms)
            
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
            
            # 並行執行
            par_time, par_norms = parallel_multiply(size)
            par_memory = torch.cuda.max_memory_allocated() / 1024**2
            
            results['parallel'][size]['times'].append(par_time)
            results['parallel'][size]['memory'].append(par_memory)
            results['parallel'][size]['norms'].append(par_norms)
            
            torch.cuda.empty_cache()
    
    return results

# 執行基準測試
results = benchmark_matrix_operations()

# 分析結果
for size in results['sequential'].keys():
    print(f"\n矩陣大小: {size}x{size}")
    
    seq_times = torch.tensor(results['sequential'][size]['times'])
    par_times = torch.tensor(results['parallel'][size]['times'])
    
    print(f"順序執行:")
    print(f"  時間: {seq_times.mean():.4f} ± {seq_times.std():.4f} 秒")
    print(f"  記憶體: {torch.tensor(results['sequential'][size]['memory']).mean():.1f} MB")
    
    print(f"並行執行:")
    print(f"  時間: {par_times.mean():.4f} ± {par_times.std():.4f} 秒")
    print(f"  記憶體: {torch.tensor(results['parallel'][size]['memory']).mean():.1f} MB")
    
    speedup = seq_times.mean() / par_times.mean()
    print(f"加速比: {speedup:.2f}x")
``````




這個改進版本：
1. 更詳細的性能指標
2. 記憶體使用分析
3. 統計誤差計算
4. 更完整的結果報告

注意事項：
1. 確保有兩個可用的GPU
2. 考慮GPU間數據傳輸開銷
3. 注意記憶體管理
4. 考慮負載均衡


練習五（新增）：

5. 如果你有 Apple Silicon Mac，比較在 MPS、CPU 上訓練一個小型神經網絡的速度差異。

我的回答：



以下是比較 MPS 和 CPU 訓練速度的完整示例：

```python
import torch
import torch.nn as nn
import torch.optim as optim
import time

def create_model():
    """創建一個簡單的神經網絡"""
    return nn.Sequential(
        nn.Linear(784, 256),
        nn.ReLU(),
        nn.Linear(256, 128),
        nn.ReLU(),
        nn.Linear(128, 10)
    )

def train_on_device(device, num_epochs=10, batch_size=64):
    """在指定設備上訓練模型"""
    # 創建模型並移動到設備
    model = create_model().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.01)
    
    # 生成假數據
    data = torch.randn(1000, 784).to(device)
    labels = torch.randint(0, 10, (1000,)).to(device)
    
    # 訓練計時
    start_time = time.time()
    
    for epoch in range(num_epochs):
        for i in range(0, len(data), batch_size):
            batch_data = data[i:i+batch_size]
            batch_labels = labels[i:i+batch_size]
            
            optimizer.zero_grad()
            outputs = model(batch_data)
            loss = criterion(outputs, batch_labels)
            loss.backward()
            optimizer.step()
        
        # 同步設備（重要！）
        if device.type == 'mps':
            torch.mps.synchronize()
        elif device.type == 'cuda':
            torch.cuda.synchronize()
    
    total_time = time.time() - start_time
    return total_time

# 測試不同設備
print("=== 訓練速度比較 ===\n")

# CPU 訓練
cpu_time = train_on_device(torch.device('cpu'))
print(f"CPU 訓練時間: {cpu_time:.4f} 秒")

# MPS 訓練（如果可用）
if torch.backends.mps.is_available():
    mps_time = train_on_device(torch.device('mps'))
    print(f"MPS 訓練時間: {mps_time:.4f} 秒")
    speedup = cpu_time / mps_time
    print(f"\nMPS 加速比: {speedup:.2f}x")
else:
    print("MPS 不可用，跳過測試")

# CUDA 訓練（如果可用）
if torch.cuda.is_available():
    cuda_time = train_on_device(torch.device('cuda'))
    print(f"\nCUDA 訓練時間: {cuda_time:.4f} 秒")
    cuda_speedup = cpu_time / cuda_time
    print(f"CUDA 加速比: {cuda_speedup:.2f}x")
```

**預期結果**：

在 Apple Silicon Mac 上：
- CPU: 基準時間
- MPS: 通常快 3-10 倍
- 具體加速比取決於模型大小和批次大小

**性能優化建議**：

1. **批次大小**：
```python
# MPS 在較大批次下性能更好
batch_sizes = [16, 32, 64, 128, 256]
for bs in batch_sizes:
    time = train_on_device(torch.device('mps'), batch_size=bs)
    print(f"批次大小 {bs}: {time:.4f} 秒")
```

2. **同步問題**：
```python
# 必須使用同步確保準確計時
if device.type == 'mps':
    torch.mps.synchronize()  # 等待所有操作完成
```

3. **內存管理**：
```python
# MPS 與系統共享記憶體
if torch.backends.mps.is_available():
    # 注意監控系統記憶體使用
    import psutil
    print(f"系統記憶體使用: {psutil.virtual_memory().percent}%")
```

**注意事項**：
- MPS 性能持續改進中，不同 PyTorch 版本性能可能不同
- 對於小模型，數據傳輸開銷可能超過計算加速
- 建議在實際工作負載上進行基準測試

練習六（新增）：

6. 嘗試將模型在 CUDA、MPS 和 CPU 之間移動，觀察數據傳輸的開銷。

我的回答：



以下是測量設備間數據傳輸開銷的完整示例：

```python
import torch
import torch.nn as nn
import time

def measure_transfer_time(tensor, from_device, to_device, num_trials=10):
    """測量張量在設備間傳輸的時間"""
    times = []
    
    for _ in range(num_trials):
        # 確保張量在源設備上
        t = tensor.to(from_device)
        
        # 同步源設備
        if from_device.type == 'cuda':
            torch.cuda.synchronize()
        elif from_device.type == 'mps':
            torch.mps.synchronize()
        
        # 計時傳輸
        start = time.time()
        t_new = t.to(to_device)
        
        # 同步目標設備
        if to_device.type == 'cuda':
            torch.cuda.synchronize()
        elif to_device.type == 'mps':
            torch.mps.synchronize()
        
        end = time.time()
        times.append(end - start)
    
    return sum(times) / len(times)

# 創建測試張量（不同大小）
sizes = [
    (100, 100, "小張量 10K 元素"),
    (1000, 1000, "中張量 1M 元素"),
    (5000, 5000, "大張量 25M 元素")
]

# 獲取可用設備
devices = [torch.device('cpu')]
if torch.cuda.is_available():
    devices.append(torch.device('cuda'))
if torch.backends.mps.is_available():
    devices.append(torch.device('mps'))

print("=== 設備間數據傳輸開銷測試 ===\n")

for size_x, size_y, desc in sizes:
    print(f"\n{desc} ({size_x}x{size_y}):")
    tensor = torch.randn(size_x, size_y)
    
    # 測試所有設備對
    for from_dev in devices:
        for to_dev in devices:
            if from_dev == to_dev:
                continue
            
            transfer_time = measure_transfer_time(tensor, from_dev, to_dev)
            print(f"  {from_dev} → {to_dev}: {transfer_time*1000:.2f} ms")

# 測試模型傳輸
print("\n=== 模型傳輸開銷測試 ===\n")

def measure_model_transfer(model, from_device, to_device):
    """測量模型在設備間傳輸的時間"""
    model = model.to(from_device)
    
    start = time.time()
    model = model.to(to_device)
    
    if to_device.type == 'cuda':
        torch.cuda.synchronize()
    elif to_device.type == 'mps':
        torch.mps.synchronize()
    
    return time.time() - start

# 創建不同大小的模型
models = [
    (nn.Linear(100, 50), "小模型"),
    (nn.Sequential(
        nn.Linear(1000, 500),
        nn.ReLU(),
        nn.Linear(500, 100)
    ), "中模型"),
    (nn.Sequential(
        nn.Linear(5000, 2000),
        nn.ReLU(),
        nn.Linear(2000, 1000),
        nn.ReLU(),
        nn.Linear(1000, 100)
    ), "大模型")
]

for model, desc in models:
    print(f"\n{desc}:")
    for from_dev in devices:
        for to_dev in devices:
            if from_dev == to_dev:
                continue
            
            transfer_time = measure_model_transfer(model, from_dev, to_dev)
            print(f"  {from_dev} → {to_dev}: {transfer_time*1000:.2f} ms")
```

**詳細分析版本**：

```python
def comprehensive_transfer_analysis():
    """全面分析數據傳輸開銷"""
    results = {}
    
    # 測試不同數據類型
    data_types = {
        'float32': torch.float32,
        'float16': torch.float16,
        'int64': torch.int64
    }
    
    size = (1000, 1000)
    devices = [torch.device('cpu')]
    if torch.cuda.is_available():
        devices.append(torch.device('cuda'))
    if torch.backends.mps.is_available():
        devices.append(torch.device('mps'))
    
    print("=== 不同數據類型的傳輸開銷 ===\n")
    
    for dtype_name, dtype in data_types.items():
        print(f"\n數據類型: {dtype_name}")
        tensor = torch.randn(size, dtype=dtype)
        
        for from_dev in devices:
            for to_dev in devices:
                if from_dev == to_dev:
                    continue
                
                time_taken = measure_transfer_time(tensor, from_dev, to_dev)
                key = f"{dtype_name}_{from_dev}_{to_dev}"
                results[key] = time_taken
                print(f"  {from_dev} → {to_dev}: {time_taken*1000:.2f} ms")
    
    return results

# 運行分析
results = comprehensive_transfer_analysis()
```

**主要發現**：

1. **傳輸方向的不對稱性**：
   - CPU → GPU 通常比 GPU → CPU 快
   - 這是由於 PCIe 總線和記憶體架構差異

2. **數據大小的影響**：
   - 小數據：傳輸開銷主導（啟動成本）
   - 大數據：帶寬限制主導

3. **數據類型的影響**：
   - float16 比 float32 傳輸快約 2 倍
   - 但需要權衡精度

**最佳實踐**：

```python
# 1. 最小化設備傳輸
# 不好的做法
for i in range(1000):
    x = x.cpu()  # 頻繁傳輸
    # 在 CPU 上處理
    x = x.cuda()

# 好的做法
# 盡量在同一設備上完成所有操作

# 2. 批量傳輸
# 不好
for tensor in tensors:
    tensor.to(device)

# 好
batch_tensor = torch.stack(tensors).to(device)

# 3. 異步傳輸（CUDA）
if torch.cuda.is_available():
    tensor = tensor.to('cuda', non_blocking=True)
```

**注意事項**：
- 傳輸開銷可能超過計算節省
- 合理規劃數據流
- 使用固定記憶體（pinned memory）可加速 CPU-GPU 傳輸

[Discussions](https://discuss.d2l.ai/t/1841)
